# Tools, Skills, and the AI as Conductor
## ME 493B — AI in Product Development | Mini-Project 3, Part A

**Instructor:** Scott Thielman, PhD — University of Washington Bothell
**Due:** Monday, May 11, 2026 at 11:59 PM
**Time estimate:** 90–120 minutes
**Points:** 50 (Part A). Part B is worth 50 points separately.

---

### What changed

In MP1 you saw what the AI knows. In MP2 you fed it what *you* know. In MP3
you give it the ability to **do things** — call your code, follow your
workflows, look at your designs. The AI stops being a chatbot and becomes
an orchestrator that uses tools you build to accomplish goals you set.

This notebook teaches you to expose capability to the AI:

1. **Function calling (tool use)** — the AI decides when to invoke your code
2. **Skill authoring** — codifying engineering judgment as a reusable artifact
3. **RAG-as-a-tool** — productizing your MP2 retrieval so the AI can call it
4. **Vision in the loop** — host-based design review (no API plumbing)
5. **Composition** — a skill that orchestrates multiple tools at once

In Part B, you'll assemble these into a workspace for the MiniClaw project.

### How visibility works in this notebook

Function calling is invisible by default — the AI calls a tool, you only see
the final text. That makes it impossible to grade and impossible to debug.

The Setup cell defines a helper, `run_with_logging()`, that wraps the API
call and prints a structured transcript every time you run a query:

```
[USER] ...
[ITERATION 1]
[TOOL CALL] convert_units({...})
[TOOL RESULT] ...
[ITERATION 2]
[AI FINAL RESPONSE] ...
```

**Sections 1, 2, 3, and 5 are graded from these transcripts.** If your
transcript shows `[TOOL CALL]` lines with the right tools and arguments,
you've done the work. **Section 4 is graded host-side** — markdown +
screenshot + response text, the same evidence pattern Part B uses for the
CAD interaction.

### Grading summary (50 pts)

| Section | Points | What the grader checks |
|---------|--------|------------------------|
| 1. Function Calling | 8 | Three transcripts show AI deciding correctly when to call the tool |
| 2. Anatomy of a Skill | 8 | Student-authored skill exists, transcript shows AI catching unit error |
| 3. Wrap Your RAG as a Tool | 10 | Tool schema printed, three transcripts, vague-vs-good comparison |
| 4. Vision in the Loop | 8 | Three documented exchanges (markdown + screenshot + response) |
| 5. Composition Capstone | 8 | Composite skill exists, with/without skill comparison |
| 6. Reflections | 8 | Three thoughtful reflections (3–4 sentences each) |
| **Total** | **50** | |

> **Reminder:** This notebook builds on MP2's GitHub Models setup. You
> already have a Personal Access Token with `models:read` scope in your
> `.env` file. If `setup` complains about credentials, refer back to your
> MP2 PAT setup notes.

---
## Section 0: Setup

Run the cells in this section **before anything else**. They install
dependencies, define `run_with_logging()`, and load the starter RAG. These
are the only pre-written code cells in the notebook — everything in
Sections 1–5 is yours to write.

In [4]:
# ============================================================
# SETUP — Run this cell first (pre-written, do not modify)
# ============================================================
import subprocess, sys

# All MP3 Part A dependencies should already be in requirements.txt from
# MP2 — but we double-check the few we depend on here.
for pkg in ["openai", "chromadb", "sentence-transformers", "python-dotenv"]:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", pkg],
        stdout=subprocess.DEVNULL,
    )

import json
import os
import textwrap
import time
from datetime import datetime
from pathlib import Path

from openai import OpenAI
from dotenv import load_dotenv

# Load .env (your GitHub PAT lives here — same as MP2)
load_dotenv()

# GitHub Models endpoint — OpenAI-compatible, free tier
GH_TOKEN = os.environ.get("GITHUB_TOKEN") or os.environ.get("GITHUB_PAT")
if not GH_TOKEN:
    raise RuntimeError(
        "Could not find GITHUB_TOKEN or GITHUB_PAT in environment. "
        "Make sure you have a .env file with GITHUB_TOKEN=ghp_... at "
        "the repo root. See your MP2 PAT setup notes."
    )

client = OpenAI(
    base_url="https://models.github.ai/inference",
    api_key=GH_TOKEN,
)

# Default model. We pin a single tool-calling SKU here. If GitHub Models
# rotates the catalog (it does, every few months), swap this for the
# current verified-stable tool-calling model — the lesson is the pattern.
DEFAULT_MODEL = "openai/gpt-4.1"

# Where transcripts get written by run_with_logging() (Sections 1, 2, 3, 5)
TRANSCRIPTS_DIR = Path("transcripts")
if not TRANSCRIPTS_DIR.exists():
    # Handle running from repo root vs. from notebook directory
    TRANSCRIPTS_DIR = Path("MP3/Part A/transcripts")
TRANSCRIPTS_DIR.mkdir(exist_ok=True)

print("Imports loaded.")
print(f"  Default model: {DEFAULT_MODEL}")
print(f"  Transcripts will be written to: {TRANSCRIPTS_DIR.resolve()}")

Imports loaded.
  Default model: openai/gpt-4.1
  Transcripts will be written to: /workspaces/ai-in-pd-spring2026/MP3/Part A/transcripts


In [5]:
# ============================================================
# run_with_logging() — the visibility mechanism for this notebook
#
# Wraps a multi-turn GitHub Models call. Executes any tool calls the AI
# makes, prints a structured transcript, and returns a dict you can
# inspect programmatically.
#
# READ THIS CAREFULLY — you'll use it in every section. Understanding
# the agent loop is part of the lesson.
# ============================================================

def run_with_logging(messages, *, tools=None, tool_handlers=None,
                     model=None, system=None, max_iterations=5,
                     verbose=True, label=None):
    """Run a multi-turn tool-using conversation, printing every step.

    Args:
        messages: list of {"role": "user"/"assistant", "content": str}
        tools: list of OpenAI-style tool schema dicts, or None
        tool_handlers: dict mapping tool name → Python callable
        model: model identifier (defaults to DEFAULT_MODEL above)
        system: optional system prompt prepended as a "system" message
        max_iterations: safety cap on agent-loop iterations
        verbose: if True, print structured transcript to cell output
        label: optional short label used in the saved JSON filename

    Returns:
        dict with keys:
            "final_message":   the AI's final text response
            "tool_calls_made": list of {"name", "args", "result"} dicts
            "iterations":      number of loop iterations used
            "transcript":      the full printed transcript as a string
    """
    model = model or DEFAULT_MODEL
    tool_handlers = tool_handlers or {}

    # Build the message list (mutates as the loop runs)
    full_messages = []
    if system:
        full_messages.append({"role": "system", "content": system})
    full_messages.extend(messages)

    # We collect printed lines into a buffer so we can save the transcript
    lines = []
    def out(s=""):
        lines.append(s)
        if verbose:
            print(s)

    bar = "═" * 62
    out(bar)
    # Print the user's question (the LAST user message — concise)
    last_user = next((m["content"] for m in reversed(messages) if m["role"] == "user"), "")
    out(f"[USER]\n{last_user}")
    out(bar)

    tool_calls_made = []
    final_text = None

    for iteration in range(1, max_iterations + 1):
        out(f"[ITERATION {iteration}]")

        kwargs = {"model": model, "messages": full_messages}
        if tools:
            kwargs["tools"] = tools
            kwargs["tool_choice"] = "auto"

        response = client.chat.completions.create(**kwargs)
        msg = response.choices[0].message

        # Did the AI ask to call any tools?
        tcs = getattr(msg, "tool_calls", None) or []

        if tcs:
            # Append the assistant message that contained the tool calls
            full_messages.append({
                "role": "assistant",
                "content": msg.content or "",
                "tool_calls": [
                    {
                        "id": tc.id,
                        "type": "function",
                        "function": {
                            "name": tc.function.name,
                            "arguments": tc.function.arguments,
                        },
                    }
                    for tc in tcs
                ],
            })

            for tc in tcs:
                name = tc.function.name
                try:
                    args = json.loads(tc.function.arguments or "{}")
                except json.JSONDecodeError:
                    args = {"_raw_arguments": tc.function.arguments}

                out(f"[TOOL CALL] {name}")
                out(f"  Args: {json.dumps(args, ensure_ascii=False)}")

                handler = tool_handlers.get(name)
                if handler is None:
                    result = {"error": f"No handler registered for tool '{name}'"}
                else:
                    try:
                        result = handler(**args)
                    except Exception as e:  # noqa: BLE001
                        result = {"error": f"{type(e).__name__}: {e}"}

                out(f"[TOOL RESULT]")
                pretty = json.dumps(result, ensure_ascii=False, indent=2)
                # Truncate long results in the printed view (full result is
                # still in the saved JSON transcript)
                if len(pretty) > 600:
                    out(textwrap.indent(pretty[:600] + " ...", "  "))
                else:
                    out(textwrap.indent(pretty, "  "))

                tool_calls_made.append({"name": name, "args": args, "result": result})

                full_messages.append({
                    "role": "tool",
                    "tool_call_id": tc.id,
                    "content": json.dumps(result, ensure_ascii=False),
                })

            out(bar)
            continue  # back to the model with the tool results in context

        # No tool calls → this is the final answer
        final_text = msg.content or ""
        out("[AI FINAL RESPONSE]")
        out(final_text)
        out(bar)
        break
    else:
        out(f"[MAX ITERATIONS HIT after {max_iterations} loops — stopping]")
        out(bar)

    out("[SUMMARY]")
    out(f"  Iterations: {iteration}")
    names = ", ".join(t["name"] for t in tool_calls_made) or "(none)"
    out(f"  Tool calls made: {len(tool_calls_made)} ({names})")
    out(f"  Final answer received: {'yes' if final_text else 'no'}")

    transcript = "\n".join(lines)

    # Save a machine-readable copy of the run
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    slug = (label or "run").lower().replace(" ", "_").replace("/", "_")
    out_path = TRANSCRIPTS_DIR / f"{slug}_{ts}.json"
    out_path.write_text(json.dumps({
        "label": label,
        "model": model,
        "messages_in": messages,
        "system": system,
        "tools_offered": [t.get("function", {}).get("name") for t in (tools or [])],
        "tool_calls_made": tool_calls_made,
        "iterations": iteration,
        "final_message": final_text,
        "transcript": transcript,
    }, ensure_ascii=False, indent=2), encoding="utf-8")

    return {
        "final_message": final_text,
        "tool_calls_made": tool_calls_made,
        "iterations": iteration,
        "transcript": transcript,
        "transcript_path": str(out_path),
    }

# Quick helper for visualizing the JSON tool schemas you'll write
def pretty_print_tool_schema(schema):
    print(json.dumps(schema, indent=2, ensure_ascii=False))

print("run_with_logging() and pretty_print_tool_schema() are defined.")

run_with_logging() and pretty_print_tool_schema() are defined.


In [6]:
# ============================================================
# Starter RAG — Ridgeline Engineering Partners corpus from MP2
# ============================================================
# The starter_rag/ folder ships with a pre-built ChromaDB collection
# (109 chunks across 20 Ridgeline documents). The query helper below is
# what you'll wrap as a tool in Section 3.
import sys
starter_dir = Path("starter_rag")
if not starter_dir.exists():
    starter_dir = Path("MP3/Part A/starter_rag")
sys.path.insert(0, str(starter_dir.resolve()))

# If the ChromaDB hasn't been built yet on this machine, build it now.
if not (starter_dir / "chroma_db").exists():
    print("Starter RAG ChromaDB not found — building it now (one-time, ~30s) ...")
    import importlib
    builder = importlib.import_module("build_starter_rag")
    builder.build()

from query_ridgeline_rag import query_ridgeline_rag

# Smoke test
smoke = query_ridgeline_rag("What is the standard billing rate for senior engineers?", n_results=2)
print("Starter RAG ready.")
print(f"  {smoke['summary']}")
print(f"  Top chunk preview: {smoke['chunks'][0]['text'][:140]!r} ...")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Starter RAG ready.
  2 chunks retrieved. Top match: DOC-014 — Standard Billing Rate Schedule — FY 2025 (score 0.6404).
  Top chunk preview: 'RIDGELINE ENGINEERING PARTNERS\nSTANDARD BILLING RATE SCHEDULE\nEffective: January 1, 2025\nApproved by: Dr. Catherine Marsh, Managing Partner\n' ...


---
## Section 1: Function Calling — How the AI Calls Your Code [8 pts]

**Concept.** Function calling is the mechanism by which the AI decides to
invoke external code. You define a tool with a JSON schema, send it
alongside the user's message, and the AI either responds normally OR
returns a structured request to call the tool. Your code executes,
returns the result, and the AI continues with the result in context.
**The AI is a decider; the tool is just Python.**

The agent loop:

```
user → AI → (optional tool call) → tool result → AI → final answer
```

Multi-turn loops are normal. The AI may call a tool, look at the result,
and decide to call another (or the same) tool again before giving you a
final answer.

### What to build

1. Wrap the provided unit-conversion math in a Python function
   `convert_units(value, from_unit, to_unit)` that returns a dict.
2. Write the JSON schema for the tool — `name`, `description`, and
   `parameters` with `type`, `properties`, `required`.
3. Run three queries through `run_with_logging()`:
   - **Query A** (should call tool): *"What's 25.4 mm in inches?"*
   - **Query B** (should NOT call tool): *"What's the typical Young's modulus of mild steel?"*
   - **Query C** (judgment call): *"I'm machining a part with a 0.001 inch
     tolerance — what's that in millimeters, and is that achievable on a
     hobby mill?"*  (needs the tool *and* domain knowledge)
4. Print a summary table comparing `tool_calls_made` vs. expectation.

### Key concepts

- The JSON schema is a **contract**. The AI reads it to decide if your
  tool is relevant.
- A good `description` is the difference between the AI calling your
  tool and ignoring it. Vague descriptions get ignored; specific ones
  get used.
- Tool choice is the AI's responsibility. You expose capability; the AI
  decides whether to use it.

### Reference links

- [OpenAI function-calling guide](https://platform.openai.com/docs/guides/function-calling)
- [GitHub Models tool use](https://docs.github.com/en/github-models)

### Proof of work [8 pts]

Three labeled `run_with_logging()` transcripts (A, B, C) printed in cell
output, a summary table that compares `tool_calls_made` to the expected
behavior, and a 2-sentence student observation: *"What changed about
the AI's behavior when you gave it a tool?"*

In [7]:
# Step 1 — Wrap the conversion math as a Python function.
#
# Use the table below. Each pair is bidirectional (mm↔inch, MPa↔psi, etc.).
# Your function should return a dict like:
#     {"value_in": <input>, "from_unit": ..., "to_unit": ...,
#      "value_out": <converted>, "factor": <multiplier>}
#
# Conversion factors (forward direction; invert for the reverse):
#     1 inch    = 25.4 mm
#     1 psi     = 0.00689476 MPa
#     1 in·lb   = 0.112985 N·m
#     1 lbf     = 4.44822 N
#     °F        = °C * 9/5 + 32
#
# Raise ValueError for unit pairs that aren't supported.
#
# Step 1 — Wrap the conversion math as a Python function.

def convert_units(value, from_unit, to_unit):
    # forward conversion factors. multiply value by factor to get target unit.
    # listed in both directions so I can just look the pair up.
    factors = {
        ("mm", "inch"):   1/25.4,
        ("inch", "mm"):   25.4,
        ("psi", "MPa"):   0.00689476,
        ("MPa", "psi"):   1/0.00689476,
        ("in_lb", "N_m"): 0.112985,
        ("N_m", "in_lb"): 1/0.112985,
        ("ft_lb", "N_m"): 1.355818,
        ("N_m", "ft_lb"): 1/1.355818,
        ("lbf", "N"):     4.44822,
        ("N", "lbf"):     1/4.44822,
    }

    # temperature is non-linear so handle separately
    if from_unit == "F" and to_unit == "C":
        out = (value - 32) * 5/9
        return {"value_in": value, "from_unit": from_unit,
                "to_unit": to_unit, "value_out": out, "factor": "nonlinear"}
    if from_unit == "C" and to_unit == "F":
        out = value * 9/5 + 32
        return {"value_in": value, "from_unit": from_unit,
                "to_unit": to_unit, "value_out": out, "factor": "nonlinear"}

    key = (from_unit, to_unit)
    if key not in factors:
        raise ValueError(f"don't know how to convert {from_unit} -> {to_unit}")

    factor = factors[key]
    return {
        "value_in":  value,
        "from_unit": from_unit,
        "to_unit":   to_unit,
        "value_out": value * factor,
        "factor":    factor,
    }



print(convert_units(25.4, "mm", "inch"))
print(convert_units(100, "lbf", "N"))
print(convert_units(212, "F", "C"))

{'value_in': 25.4, 'from_unit': 'mm', 'to_unit': 'inch', 'value_out': 0.9999999999999999, 'factor': 0.03937007874015748}
{'value_in': 100, 'from_unit': 'lbf', 'to_unit': 'N', 'value_out': 444.822, 'factor': 4.44822}
{'value_in': 212, 'from_unit': 'F', 'to_unit': 'C', 'value_out': 100.0, 'factor': 'nonlinear'}


In [8]:
# Step 2 — Write the JSON schema for convert_units.
#
# The schema is a Python dict that follows the OpenAI tool format:
#     {
#       "type": "function",
#       "function": {
#         "name": ...,
#         "description": ...,   # be specific — this is what the AI reads
#         "parameters": {
#           "type": "object",
#           "properties": {
#             "value": {"type": "number", "description": "..."},
#             "from_unit": {"type": "string", "description": "..."},
#             "to_unit":   {"type": "string", "description": "..."},
#           },
#           "required": ["value", "from_unit", "to_unit"]
#         }
#       }
#     }
#
# Step 2 — JSON schema for convert_units.

convert_units_tool = {
    "type": "function",
    "function": {
        "name": "convert_units",
        "description": (
            "Convert an engineering quantity between unit systems. "
            "Supports length (mm <-> inch), pressure/stress (psi <-> MPa), "
            "torque (in_lb <-> N_m, ft_lb <-> N_m), force (lbf <-> N), and "
            "temperature (F <-> C). Use whenever the user provides a "
            "value in one unit system and needs it expressed in another."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "value": {
                    "type": "number",
                    "description": "Numeric value to convert.",
                },
                "from_unit": {
                    "type": "string",
                    "description": "Source unit. One of: mm, inch, psi, MPa, in_lb, ft_lb, N_m, lbf, N, F, C.",
                },
                "to_unit": {
                    "type": "string",
                    "description": "Target unit, same options as from_unit.",
                },
            },
            "required": ["value", "from_unit", "to_unit"],
        },
    },
}

pretty_print_tool_schema(convert_units_tool)

{
  "type": "function",
  "function": {
    "name": "convert_units",
    "description": "Convert an engineering quantity between unit systems. Supports length (mm <-> inch), pressure/stress (psi <-> MPa), torque (in_lb <-> N_m, ft_lb <-> N_m), force (lbf <-> N), and temperature (F <-> C). Use whenever the user provides a value in one unit system and needs it expressed in another.",
    "parameters": {
      "type": "object",
      "properties": {
        "value": {
          "type": "number",
          "description": "Numeric value to convert."
        },
        "from_unit": {
          "type": "string",
          "description": "Source unit. One of: mm, inch, psi, MPa, in_lb, ft_lb, N_m, lbf, N, F, C."
        },
        "to_unit": {
          "type": "string",
          "description": "Target unit, same options as from_unit."
        }
      },
      "required": [
        "value",
        "from_unit",
        "to_unit"
      ]
    }
  }
}


In [9]:
# Step 3 — Query A: should call the tool
#
# Use run_with_logging() with:
#   messages       = [{"role": "user", "content": "What's 25.4 mm in inches?"}]
#   tools          = [convert_units_tool]
#   tool_handlers  = {"convert_units": convert_units}
#   label          = "section1_queryA"
#
# Inspect the transcript: did the AI call convert_units?
# result_A = run_with_logging( ... )
result_A = run_with_logging(
    messages=[{"role": "user", "content": "What's 25.4 mm in inches?"}],
    tools=[convert_units_tool],
    tool_handlers={"convert_units": convert_units},
    label="section1_queryA",
)
 

══════════════════════════════════════════════════════════════
[USER]
What's 25.4 mm in inches?
══════════════════════════════════════════════════════════════
[ITERATION 1]
[TOOL CALL] convert_units
  Args: {"from_unit": "mm", "to_unit": "inch", "value": 25.4}
[TOOL RESULT]
  {
    "value_in": 25.4,
    "from_unit": "mm",
    "to_unit": "inch",
    "value_out": 0.9999999999999999,
    "factor": 0.03937007874015748
  }
══════════════════════════════════════════════════════════════
[ITERATION 2]
[AI FINAL RESPONSE]
25.4 mm is equal to 1 inch.
══════════════════════════════════════════════════════════════
[SUMMARY]
  Iterations: 2
  Tool calls made: 1 (convert_units)
  Final answer received: yes


In [10]:
# Step 3 — Query B: should NOT call the tool
# Same setup, different question:
#   "What's the typical Young's modulus of mild steel?"
# label = "section1_queryB"
result_B = run_with_logging(
    messages=[{"role": "user",
               "content": "What's the typical Young's modulus of mild steel?"}],
    tools=[convert_units_tool],
    tool_handlers={"convert_units": convert_units},
    label="section1_queryB",
)
 

══════════════════════════════════════════════════════════════
[USER]
What's the typical Young's modulus of mild steel?
══════════════════════════════════════════════════════════════
[ITERATION 1]
[AI FINAL RESPONSE]
The typical Young’s modulus (also known as the modulus of elasticity) of mild steel is approximately **200 GPa** (gigapascals), which is equivalent to **29,000 ksi** (kilopounds per square inch), or **29 x 10^6 psi** (pounds per square inch).

This value can vary slightly depending on the exact composition and processing of the steel, but 200 GPa (or 29,000 ksi) is commonly used in engineering calculations for structural steel.
══════════════════════════════════════════════════════════════
[SUMMARY]
  Iterations: 1
  Tool calls made: 0 ((none))
  Final answer received: yes


In [11]:
# Step 3 — Query C: judgment call (tool + domain knowledge)
# Question:
#   "I'm machining a part with a 0.001 inch tolerance — what's that in
#    millimeters, and is that achievable on a hobby mill?"
# label = "section1_queryC"
result_C = run_with_logging(
    messages=[{"role": "user", "content":
        "I'm machining a part with a 0.001 inch tolerance — what's that in "
        "millimeters, and is that achievable on a hobby mill?"}],
    tools=[convert_units_tool],
    tool_handlers={"convert_units": convert_units},
    label="section1_queryC",
)

══════════════════════════════════════════════════════════════
[USER]
I'm machining a part with a 0.001 inch tolerance — what's that in millimeters, and is that achievable on a hobby mill?
══════════════════════════════════════════════════════════════
[ITERATION 1]
[TOOL CALL] convert_units
  Args: {"from_unit": "inch", "to_unit": "mm", "value": 0.001}
[TOOL RESULT]
  {
    "value_in": 0.001,
    "from_unit": "inch",
    "to_unit": "mm",
    "value_out": 0.0254,
    "factor": 25.4
  }
══════════════════════════════════════════════════════════════
[ITERATION 2]
[AI FINAL RESPONSE]
A 0.001 inch tolerance equals 0.0254 millimeters.

Regarding achievability: A 0.001 inch (0.0254 mm) tolerance is extremely tight for most hobby milling machines. While some higher-end hobby mills and skilled machinists can occasionally achieve this, maintaining that consistently across parts is very challenging. It usually requires precision equipment, careful temperature control, and skillful measurement pra

In [12]:
# Step 4 — Summary table.
#
# Print something like:
#   Query A (mm to inches):           tool_calls_made = 1   [expected: 1]
#   Query B (Young's modulus):        tool_calls_made = 0   [expected: 0]
#   Query C (tolerance + judgment):   tool_calls_made = ?   [expected: 1]
#
# Use the result["tool_calls_made"] lists from the runs above.
a_calls = [tc["name"] for tc in result_A["tool_calls_made"]]
b_calls = [tc["name"] for tc in result_B["tool_calls_made"]]
c_calls = [tc["name"] for tc in result_C["tool_calls_made"]]
 
print("Query                              calls   expected   tools")
print("-" * 70)
print(f"A (25.4 mm to inches)              {len(a_calls)}       1          {a_calls}")
print(f"B (Youngs modulus of mild steel)   {len(b_calls)}       0          {b_calls}")
print(f"C (0.001 in tolerance + judgment)  {len(c_calls)}       1          {c_calls}")

Query                              calls   expected   tools
----------------------------------------------------------------------
A (25.4 mm to inches)              1       1          ['convert_units']
B (Youngs modulus of mild steel)   0       0          []
C (0.001 in tolerance + judgment)  1       1          ['convert_units']


### ✏️ YOUR OBSERVATION

*In 2–3 sentences, answer:* what changed about the AI's behavior when
you gave it a tool? Did anything surprise you about Query C?

Giving the AI a tool changed it from always answering directly to deciding whether to delegate. On Query A it called convert_units with the mm-to-inch arguments and used the returned value in its response, while on Query B it skipped the tool and answered Young's modulus from training, which was the right call since the question wasn't a conversion. What I noticed on Query C is that the AI handled the conversion and the engineering judgment about hobby mill tolerances in one response, and the tool gave it an auditable number to anchor the rest of the answer on.

> **Forward connection.** The same mechanism scales to any Python
> function — calculation, file I/O, web requests, database queries. In
> Section 3 you'll wrap your RAG retrieval as a tool and watch the AI
> choose to query your engineering knowledge base. In Section 5 you'll
> compose this unit converter with the RAG tool to handle questions
> that need both.

---
## Section 2: Anatomy of a Skill [8 pts]

**Concept.** A *skill* is a reusable orchestration artifact — a markdown
file that tells an AI how to handle a class of tasks. Skills capture
the "how to do this kind of work" knowledge that an experienced
engineer has but doesn't write down. By codifying it, you make that
judgment available to **any AI session, by anyone**, without you being
there.

Skills sit *between* system prompts and tools:

- **Above tools** — they orchestrate which tools to call when
- **Below system prompts** — they're reusable across sessions; system
  prompts vary per session

The format isn't standardized across the industry yet. Anthropic uses
`SKILL.md`, OpenAI uses Custom GPT instructions, Cursor uses
`.cursorrules`, GitHub Copilot uses `.github/copilot-instructions.md`.
The *idea* is the same — codified, reusable orchestration.

### What to build

1. **Read** `SKILL_me493b_course_structure.md` at the repo root. Identify
   its anatomy (front matter, "when to use," structured guidance,
   instructions, anti-patterns).
2. **Read** the provided specimen `skills/example_design_review.md`.
   Compare its structure to the course skill.
3. **Write your own** small skill: `skills/unit_check_skill.md`. The
   skill should help an AI catch unit-conversion errors in engineering
   text. Required elements:
   - YAML front matter (`name`, `description`)
   - "When to use this skill" — the trigger conditions
   - "Steps" — the workflow (3–5 steps)
   - "What to flag" — specific patterns to watch for
   - "What NOT to do" — at least one anti-pattern
4. **Test it.** Load the skill text as a system prompt and send a test
   message containing a unit error:
   > *"The shaft transmits 50 N·m of torque with a safety factor of
   > 2.5, requiring a diameter of 12 inches per Lewis bending equation."*
   Run via `run_with_logging()`. Did the AI catch the unit issue?

### Key concepts

- Skills are NOT prompts. A prompt is what the user types now. A skill
  is what the AI knows about how to handle this *class* of task
  whenever it comes up.
- A good skill is the difference between *"the AI can handle this if I
  prompt carefully"* and *"any junior engineer can fire up Copilot and
  have the AI handle this correctly."*

### Proof of work [8 pts]

The text of your `unit_check_skill.md` printed via `Path.read_text()`,
a `run_with_logging()` transcript showing the AI being given the skill
as system prompt and catching the unit error, and a 2-sentence
observation contrasting "skill" vs. "prompt every time."

In [16]:
# Step 1 — Inspect the course skill at the repo root.
# Print the first ~40 lines and skim its structure.
from pathlib import Path
course_skill_path = Path("../../SKILL_me493b_course_structure.md")
if not course_skill_path.exists():
    course_skill_path = Path("SKILL_me493b_course_structure.md")
print("\n".join(course_skill_path.read_text(encoding="utf-8").splitlines()[:40]))

---
name: me493b-course-structure
description: >
  Reference for all structural, naming, and repository conventions for
  ME 493B "AI in Product Development" at UW Bothell (Spring 2026).
  Consult this file before creating any course file, notebook, folder,
  or spec document. Covers repo layout, file naming, notebook structure,
  spec document format, GitHub Classroom workflow, Canvas integration,
  and Copilot interaction patterns.
---

# ME 493B Course Structure Reference

## Course identity
- **Course:** ME 493B — AI in Product Development
- **Institution:** University of Washington Bothell
- **Term:** Spring 2026 (March 30 – June 5, 20 sessions, TTh 3:30–5:30 PM)
- **Instructor:** Scott Thielman, PhD (thielman@uw.edu)
- **Room:** UW1 041
- **Enrollment:** 24 students max
- **Credits:** 4

---

## Course architecture

### The Five Pillars (recurring vocabulary and rubric dimensions)

| Pillar | Short definition |
|--------|----------------|
| **Goal & Direction** | Specifying inten

In [17]:
# Step 2 — Read the specimen skill.
specimen_path = Path("skills/example_design_review.md")
if not specimen_path.exists():
    specimen_path = Path("MP3/Part A/skills/example_design_review.md")
print(specimen_path.read_text(encoding="utf-8"))

---
name: 3d-printed-part-design-review
description: >
  Review a 3D-printable mechanical part for FDM-printability, structural
  adequacy, and annotation hygiene before the part is sent to a print queue.
  Use when a user pastes a sketch, render, drawing, or description of a part
  that will be printed in PLA, PETG, or similar FDM material on a desktop
  machine (Prusa-class, Bambu-class, hobby Ender-class).
---

# Skill: 3D-Printed Part Design Review

## When to use this skill

Trigger this skill any time the user presents a part — image, screenshot,
verbal description, or CAD export — and asks for a review, critique, or
"what's wrong with this." Use it whenever the part will be printed via FDM.
Do NOT trigger for SLA, SLS, or machined parts; the constraints are
different.

## Steps

1. **Identify the part class.** Bracket, gear, housing, snap-fit, jig,
   hand tool, decorative? Different classes have different failure modes.
2. **Walk the print orientation.** What direction will the

### ✏️ YOUR TURN — Write `skills/unit_check_skill.md`

Create the file at `skills/unit_check_skill.md`. Include all five
required elements (front matter, when-to-use, steps, what-to-flag,
what-not-to-do). Aim for ~30–60 lines — sharp and specific beats long
and vague.

The next code cell prints your file back so the grader can see it. If
it errors with "FileNotFoundError," you haven't created the file yet.

In [19]:
# Step 3 — Print your skill back so it's part of the cell output.
skill_path = Path("skills/unit_check_skill.md")
if not skill_path.exists():
    skill_path = Path("MP3/Part A/skills/unit_check_skill.md")

print("=" * 62)
print(f"skills/unit_check_skill.md ({skill_path.resolve()})")
print("=" * 62)
if skill_path.exists():
    print(skill_path.read_text(encoding="utf-8"))
else:
    print("(file does not exist yet — create it, then re-run this cell)")

skills/unit_check_skill.md (/workspaces/ai-in-pd-spring2026/MP3/Part A/skills/unit_check_skill.md)
Save this exact content as MP3/Part A/skills/unit_check_skill.md. First character is a dash, last character is a period.

name: unit-check
description: >
Review a piece of engineering text (a calculation walk-through, a memo,
a problem statement) for unit-of-measure errors before the numbers are
used. Triggered when the user asks you to check or verify an
engineering calculation that involves physical quantities with units.
Skill: Engineering Unit Check
When to use this skill
Use this whenever the user shares engineering text that mixes numbers
and units — torque + shaft diameter sizing, beam stress + safety factor,
fastener torque specs, pressure drops, anything where the answer
depends on the units coming out right. Use it even if the user didn't
explicitly ask for a "unit review" — if the message contains a
calculation and units, this skill applies. Don't use it for pure
symbolic algeb

In [22]:
# Step 4 — Test the skill. Load it as a system prompt and send the test
# message. The AI should catch the unit error.
#
test_message = (
    "The shaft transmits 50 N·m of torque with a safety factor of 2.5, "
    "requiring a diameter of 12 inches per the Lewis bending equation."
)

result_skill_test = run_with_logging(
    messages=[{"role": "user", "content": test_message}],
    system=skill_path.read_text(encoding="utf-8"),
    label="section2_skill_test",
)

══════════════════════════════════════════════════════════════
[USER]
The shaft transmits 50 N·m of torque with a safety factor of 2.5, requiring a diameter of 12 inches per the Lewis bending equation.
══════════════════════════════════════════════════════════════
[ITERATION 1]
[AI FINAL RESPONSE]
1. "Diameter of 12 inches": The diameter is specified in inches (US customary) while the torque is given in N·m (SI). Mixing unit systems is a critical error. The Lewis equation and related shaft calculations require consistent units throughout.
   - Corrected version: If the torque is in N·m, diameter should be in meters or millimeters (e.g., 0.305 m or 305 mm for 12 inches) or convert torque to ft·lb if keeping diameter in inches.

2. "50 N·m of torque": Torque is specified in SI units, but the subsequent calculation uses US customary units for diameter. No conversion factor is shown.
   - Corrected version: Either convert 50 N·m to in·lb (50 N·m ≈ 442 in·lb) to use with diameter in inches,

### ✏️ YOUR OBSERVATION

*In 2–3 sentences:* what's the difference between writing this as a
skill vs. just typing the instructions in your prompt every time?

Writing this as a skill instead of a prompt made the AI's review consistent with the steps the skill spells out. In the test run the AI flagged the diameter-of-12-inches as a unit-system mismatch (US customary diameter with SI torque), pointed out that the Lewis bending equation is for gear teeth and not shaft diameter sizing, and ended with the exact "unit issue — correctable" verdict the skill's output format asks for. The difference from a prompt is that the same review pattern is now reusable and a teammate can drop the markdown into their own session and get the same structured response without me being there to prompt for it.

> **Forward connection.** In Section 5 you'll write a skill that
> orchestrates *multiple* tools. In Part B, your MiniClaw skills will
> be loaded by an AI host (Copilot agent mode, Claude Desktop, Cursor)
> with your MCP server connected — when a teammate asks a question
> about MiniClaw, your skill will guide the response.

---
## Section 3: Wrap Your RAG as a Tool [10 pts] — *centerpiece*

**Concept.** Your MP2 RAG is currently a Python function you call
directly. To make it AI-callable, you wrap it as a **tool** with a
JSON schema and connect it to the agent loop. The result: an AI that
can answer questions about Ridgeline's project history because it
knows to query your retrieval system *on demand*. This is the same
pattern that powers Copilot's codebase search, Claude's web search,
and every enterprise RAG product.

### What to build

1. The starter RAG provides `query_ridgeline_rag(question, n_results=3)`
   (already imported in the Setup section). If your MP2 RAG is fully
   working you can substitute it instead — but the starter is fine.
2. Wrap it as a tool with a JSON schema. **Critical:** the
   `description` field must be specific enough that the AI knows when
   to use it. Compare:
   - **Bad:** `"description": "Searches a database."`
   - **Good:** `"description": "Searches Ridgeline Engineering
     Partners' internal knowledge base — past project summaries,
     billing rates, technical standards, employee policies, FEA
     guidelines, and material selection reports. Use this when the
     user asks about Ridgeline's specific projects, internal
     practices, or company-specific information."`
3. Run three queries via `run_with_logging()`:
   - **Q1:** *"What was the rating factor on the Millbrook County bridge project?"*
   - **Q2:** *"What's the difference between cast iron and ductile iron?"*
     (general knowledge — should NOT call the tool)
   - **Q3:** *"What FEA mesh refinement standard does Ridgeline use,
     and is that consistent with industry best practice?"*
     (should call the tool for the first half, answer from training
     for the second half)
4. **Failure-mode exercise:** rewrite the tool description to be
   deliberately vague (e.g., `"Looks things up."`). Re-run Q1.
   Observe whether the AI still calls the tool. Restore the good
   description afterwards.
5. Build a summary table comparing `tool_called` vs. expected.

### Key concepts

- Tool description quality is the **dominant variable** in tool
  selection. Treat it like a Stack Overflow tag — specific enough that
  the right questions land here.
- The AI does NOT verify the tool's output. If your retrieval returns
  garbage, the AI will confidently use the garbage. Eval & Trust
  applies to tool results, not just final answers.
- Multi-turn behavior matters: the AI may call the tool, get a partial
  answer, and call it AGAIN with a refined query. That's the agent
  loop working.

### Proof of work [10 pts]

The full tool schema printed (good description), three labeled
transcripts for Q1/Q2/Q3, the failure-mode comparison transcript with
a vague description, a summary table, and a 3-sentence observation:
*"When did the AI choose to use your tool? When did it skip it? What
does this tell you about how to write tools for engineering work?"*

In [23]:
# Step 1 — Sanity-check that you can call the RAG directly.
# Print one chunk so you've seen the shape of the data.
sample = query_ridgeline_rag("standard billing rate for senior engineers", n_results=2)
print(sample["summary"])
for c in sample["chunks"]:
    print(f"  [{c['doc_id']}] {c['title']} (score {c['score']})")

2 chunks retrieved. Top match: DOC-014 — Standard Billing Rate Schedule — FY 2025 (score 0.6053).
  [DOC-014] Standard Billing Rate Schedule — FY 2025 (score 0.6053)
  [DOC-004] Employee Handbook Excerpts — Training and Development (score 0.4856)


In [25]:
# Step 2 — Write the tool schema for query_ridgeline_rag.
#
# Tool schema for the Ridgeline RAG.
# Description names the company, the document types, and tells the AI when NOT
# to use it. The bullet point in Section 3 said vague descriptions get ignored
# so I tried to make this pretty specific.

rag_tool_good = {
    "type": "function",
    "function": {
        "name": "query_ridgeline_rag",
        "description": (
            "Search Ridgeline Engineering Partners' internal knowledge base. "
            "Indexes 20 internal documents including project summaries "
            "(Millbrook County bridge, Cascade thermal study, Greenfield "
            "Plaza HVAC, CardioSense enclosure), the FY2025 billing rate "
            "schedule, employee handbook excerpts, technical standards, "
            "the FEA best-practices guide, tolerance analysis procedure, "
            "and material selection reports. "
            "Use this when the user asks about a specific Ridgeline project, "
            "an internal Ridgeline procedure, or a company billing rate or "
            "standard. Do NOT use this for general engineering knowledge "
            "(material properties, industry-wide practices) that doesn't "
            "depend on Ridgeline's internal documents."
        ),
        "parameters": {
            "type": "object",
            "properties": {
                "question": {
                    "type": "string",
                    "description": "Question to search the corpus with.",
                },
                "n_results": {
                    "type": "integer",
                    "description": "How many chunks to return. Default 3, max 10.",
                },
            },
            "required": ["question"],
        },
    },
}

pretty_print_tool_schema(rag_tool_good)

{
  "type": "function",
  "function": {
    "name": "query_ridgeline_rag",
    "description": "Search Ridgeline Engineering Partners' internal knowledge base. Indexes 20 internal documents including project summaries (Millbrook County bridge, Cascade thermal study, Greenfield Plaza HVAC, CardioSense enclosure), the FY2025 billing rate schedule, employee handbook excerpts, technical standards, the FEA best-practices guide, tolerance analysis procedure, and material selection reports. Use this when the user asks about a specific Ridgeline project, an internal Ridgeline procedure, or a company billing rate or standard. Do NOT use this for general engineering knowledge (material properties, industry-wide practices) that doesn't depend on Ridgeline's internal documents.",
    "parameters": {
      "type": "object",
      "properties": {
        "question": {
          "type": "string",
          "description": "Question to search the corpus with."
        },
        "n_results": {
         

In [26]:
# Step 3 — Q1: Should call the tool
#   "What was the rating factor on the Millbrook County bridge project?"
#
tool_handlers = {"query_ridgeline_rag": query_ridgeline_rag}

result_q1 = run_with_logging(
    messages=[{"role": "user",
               "content": "What was the rating factor on the Millbrook County bridge project?"}],
    tools=[rag_tool_good],
    tool_handlers=tool_handlers,
    label="section3_q1_millbrook",
)

══════════════════════════════════════════════════════════════
[USER]
What was the rating factor on the Millbrook County bridge project?
══════════════════════════════════════════════════════════════
[ITERATION 1]
[TOOL CALL] query_ridgeline_rag
  Args: {"question": "What was the rating factor on the Millbrook County bridge project?"}
[TOOL RESULT]
  {
    "chunks": [
      {
        "text": "PROJECT SUMMARY REPORT\nProject: Millbrook County DOT Bridge Load Rating Assessment\nProject Number: REP-2023-047\nClient: Millbrook County Department of Transportation\nProject Manager: Dr. Sarah Hendricks, PE, PhD\nDate Completed: August 2024\nBudget: $285,000 | Actual: $271,400\n\nSCOPE\nRidgeline was engaged by the Millbrook County DOT to perform a comprehensive load rating assessment of the Cedar Creek Bridge (Bridge No. MC-4412), a three-span continuous steel plate girder bridge constructed in 1987",
        "doc_id": "DOC-006",
        "title": "Project  ...
════════════════════════════════

In [27]:
# Step 3 — Q2: Should NOT call the tool (general knowledge)
result_q2 = run_with_logging(
    messages=[{"role": "user",
               "content": "What's the difference between cast iron and ductile iron?"}],
    tools=[rag_tool_good],
    tool_handlers=tool_handlers,
    label="section3_q2_cast_iron",
)

══════════════════════════════════════════════════════════════
[USER]
What's the difference between cast iron and ductile iron?
══════════════════════════════════════════════════════════════
[ITERATION 1]
[AI FINAL RESPONSE]
**Cast iron** and **ductile iron** are both alloys primarily composed of iron and carbon, but their properties and applications differ significantly due to differences in their microstructure:

**Cast Iron:**
- **Structure:** Traditional cast iron (such as gray cast iron) contains carbon in the form of flake graphite.
- **Properties:** Brittle, high hardness, good wear resistance, but not very ductile. The flakes create points of weakness, making cast iron prone to cracking under tension or impact.
- **Uses:** Engine blocks, machinery bases, pipes, cookware, and decorative items.

**Ductile Iron (also known as Nodular or Spheroidal Graphite Iron):**
- **Structure:** Carbon is present as spheroidal (round) graphite nodules instead of flakes. This is achieved by addi

In [28]:
# Step 3 — Q3: Multi-turn judgment call
result_q3 = run_with_logging(
    messages=[{"role": "user",
               "content": "What FEA mesh refinement standard does Ridgeline use, "
                          "and is that consistent with industry best practice?"}],
    tools=[rag_tool_good],
    tool_handlers=tool_handlers,
    label="section3_q3_fea_mesh",
)

══════════════════════════════════════════════════════════════
[USER]
What FEA mesh refinement standard does Ridgeline use, and is that consistent with industry best practice?
══════════════════════════════════════════════════════════════
[ITERATION 1]
[TOOL CALL] query_ridgeline_rag
  Args: {"question": "FEA mesh refinement standard used by Ridgeline"}
[TOOL RESULT]
  {
    "chunks": [
      {
        "text": "RIDGELINE ENGINEERING PARTNERS — FEA BEST PRACTICES GUIDE\nDocument Number: ENG-FEA-BP-01, Revision 4\nEffective Date: April 2024\nAuthor: Dr. Sarah Hendricks, PE, PhD\nApproved by: Brian Tanaka, PE, Director of Engineering\n\nThis guide establishes minimum requirements for finite element analysis performed at Ridgeline. All FEA work must comply with these practices regardless of software platform (ANSYS, Abaqus, NASTRAN, CSiBridge, COMSOL, etc.).\n\n1. MODEL PLANNING\nBefore constructing any FEA model, ",
        "doc_id": "DOC-017",
        "title": "FEA Best ...
═════════════

In [29]:
# Step 4 — Failure-mode exercise.
#
# Make a copy of your tool schema with a deliberately vague description
# (something like "Looks things up.") and re-run Q1. Compare to the
# good-description run.
#
rag_tool_vague = {
    "type": "function",
    "function": {
        "name": "query_ridgeline_rag",
        "description": "Looks things up.",
        "parameters": rag_tool_good["function"]["parameters"],
    },
}

result_q1_vague = run_with_logging(
    messages=[{"role": "user",
               "content": "What was the rating factor on the Millbrook County bridge project?"}],
    tools=[rag_tool_vague],
    tool_handlers=tool_handlers,
    label="section3_q1_vague_description",
)

══════════════════════════════════════════════════════════════
[USER]
What was the rating factor on the Millbrook County bridge project?
══════════════════════════════════════════════════════════════
[ITERATION 1]
[TOOL CALL] query_ridgeline_rag
  Args: {"question": "What was the rating factor on the Millbrook County bridge project?"}
[TOOL RESULT]
  {
    "chunks": [
      {
        "text": "PROJECT SUMMARY REPORT\nProject: Millbrook County DOT Bridge Load Rating Assessment\nProject Number: REP-2023-047\nClient: Millbrook County Department of Transportation\nProject Manager: Dr. Sarah Hendricks, PE, PhD\nDate Completed: August 2024\nBudget: $285,000 | Actual: $271,400\n\nSCOPE\nRidgeline was engaged by the Millbrook County DOT to perform a comprehensive load rating assessment of the Cedar Creek Bridge (Bridge No. MC-4412), a three-span continuous steel plate girder bridge constructed in 1987",
        "doc_id": "DOC-006",
        "title": "Project  ...
════════════════════════════════

In [30]:
# Step 5 — Summary table.
#
def rag_called(result):
    for tc in result["tool_calls_made"]:
        if tc["name"] == "query_ridgeline_rag":
            return True
    return False

print("Run                                  called?   expected")
print("-" * 60)
print(f"Q1 Millbrook (good description)      {rag_called(result_q1)}     yes")
print(f"Q1 Millbrook (VAGUE description)     {rag_called(result_q1_vague)}     yes (but description is bad)")
print(f"Q2 cast vs ductile iron              {rag_called(result_q2)}    no")
print(f"Q3 FEA mesh + industry practice      {rag_called(result_q3)}     yes")

Run                                  called?   expected
------------------------------------------------------------
Q1 Millbrook (good description)      True     yes
Q1 Millbrook (VAGUE description)     True     yes (but description is bad)
Q2 cast vs ductile iron              False    no
Q3 FEA mesh + industry practice      True     yes


### ✏️ YOUR OBSERVATION

*In 3 sentences:* When did the AI choose to use your tool? When did it
skip it? What does this tell you about how to write tools for
engineering work?

The AI called the RAG tool on Q1 (Millbrook bridge) and Q3 (Ridgeline's FEA standard) because both questions named a Ridgeline-specific artifact, and it cleanly skipped the tool on Q2 (cast vs ductile iron) because that's textbook material. The interesting run was Q1 with the "Looks things up" description and the AI still called the tool and returned the 0.73 rating factor, which surprised me until I realized the question itself ("Millbrook County bridge project") was specific enough to make the tool obviously relevant even with a useless description. For engineering tools the lesson is that description quality matters most when the user's question is generic; a sharp description that names the company, document types, and trigger conditions is what lets the tool get picked when the user's wording wouldn't have surfaced it on its own.

> **Forward connection.** In Part B you'll wrap your *MiniClaw* RAG as
> an MCP server tool that Copilot or Claude Desktop can call from your
> IDE. The principle is identical to what you just did — only the
> wire protocol changes.

---
## Section 4: Vision in the Loop — Host-Based Design Review [8 pts]

**Concept.** Engineering work is visual. CAD models, sketches, hand
drawings, prototype photos, render captures — most engineering
judgment happens *by looking at things*. Modern AI hosts (Copilot Chat,
Claude.ai, Claude Desktop, Cursor, ChatGPT) all accept image input
directly in the chat interface. This is the workflow you'll actually
use as a working engineer: paste a screenshot, ask for review, decide
what to act on. **No API plumbing required.**

The proof-of-work for this section is a *documented exchange* —
markdown + screenshot + response text. Same evidence pattern Part B
uses for the CAD interaction. Section 4 is your warm-up.

### What to do

1. Open the AI host of your choice. **Copilot Chat in VS Code is the
   default** — it's already installed and you've been using it. Other
   acceptable hosts: Claude.ai (web), Claude Desktop, Cursor,
   ChatGPT. Pick one and note it.
2. Load `images/bracket_sketch.png` (a hand-sketched L-bracket with
   deliberate errors). Paste it into the host's chat.
3. Send the prompt:

   > *"You are reviewing a hand sketch for an L-bracket that will be
   > 3D printed in PLA. Identify any issues you see — dimensional,
   > manufacturing, structural, or annotation. Be specific."*
4. Capture three things:
   - A markdown cell with the prompt you sent
   - A screenshot of the host UI showing the image and the response,
     saved to `section4_screenshots/sketch_review.png` (or similar)
   - A markdown cell with the response text copied in
5. Repeat with `images/openscad_render.png` (a render with a visible
   undercut that won't print well). Same prompt, same evidence
   capture.
6. **Comparative exercise.** Open a fresh chat and ask the same review
   question with NO image — just a text description: *"I have a
   hand-sketched L-bracket for 3D printing in PLA. Annotated
   dimensions, two M4 holes, gusset on the back. What issues should I
   check for?"* Capture the response. Compare to the image-based
   reviews.

### Key concepts

- This is the workflow you'll actually use day-to-day, and it's
  exactly what Part B's CAD interaction deliverable asks for.
- Hosts handle image encoding, model selection, and API plumbing for
  you. Your job is the engineering question and the engineering
  judgment.
- The AI's vision is good but not perfect. It will sometimes
  hallucinate features, miss subtle issues, or misread numbers. This
  is exactly where Eval & Trust applies — *the engineer judges what
  to act on.*
- Different hosts use different vision-capable models with different
  strengths. If a review feels thin, try a second host.

### Proof of work [8 pts]

For each of the **three exchanges** (sketch image, CAD render image,
text-only baseline), commit:
- The prompt as markdown
- A screenshot in `section4_screenshots/`
- The response text as markdown

Plus the comparison table below — fill in the issue catalog and the
counts.

### Exchange 1 — Hand sketch (bracket_sketch.png)

**Host used:** chatgpt

**Prompt I sent:**

You are reviewing a hand sketch for an L-bracket that will be 3D printed in PLA. Identify any issues you see which is dimensional, manufacturing, structural, or annotation. Be specific.

**Screenshot:** `section4_screenshots/sketch_review.png`
![alt text](image.png)

_Embed the screenshot below by uncommenting the next line:_
`<!-- ![Sketch review](section4_screenshots/sketch_review.png) -->`

**Response (copied from host):**

Yeah, there are a lot of red flags in this sketch. Some are “rough sketch” issues, but several would cause manufacturing, strength, or inspection problems.

Major dimensional issues
1. Mixed units without a clear standard

The sketch uses 80 mm, 32 mm, Ø4 mm, 0.3 mm, and also 2.36 in.

That is a problem because the drawing never states whether the part is controlled in metric or inch units. The note says:

ALL DIMS ±0.001" unless noted

That tolerance is in inches, but most dimensions are in millimeters. This creates confusion during manufacturing and inspection.

A cleaner note would be something like:

UNLESS OTHERWISE SPECIFIED: DIMENSIONS ARE IN mm. TOLERANCE ±0.25 mm.

For an FDM printed PLA part, ±0.001 in is unrealistic anyway.

Manufacturing issues
2. Wall thickness of 0.3 mm is extremely thin

A 0.3 mm wall is basically not practical for most FDM prints.

Typical FDM nozzle diameter is around 0.4 mm, so a 0.3 mm wall may be thinner than one extrusion line. Even if the slicer tries to print it, the wall will likely be weak, inconsistent, or not print at all.

For PLA FDM, a more reasonable minimum wall would be around:

1.2 mm minimum, or about 3 perimeters with a 0.4 mm nozzle

For a structural bracket, I would go thicker:

2 mm to 4 mm, depending on the load.

3. No thickness/depth dimension is given

The sketch shows the L bracket from the side, but it does not specify the part thickness into the page.

For a 3D printed bracket, this is critical.

You need a dimension such as:

Bracket width/depth = 15 mm, 20 mm, etc.

Without that, the part cannot be fully modeled or manufactured.

4. The inside corner has a sharp 90° transition

The inside corner of the L shape appears sharp. That is bad structurally and not ideal for printing.

Sharp internal corners create stress concentrations, especially if this bracket is loaded in bending. Add a fillet or gusset.

Better:

Internal fillet radius: R3 mm to R8 mm

or add a triangular rib/gusset between the vertical and horizontal legs.

The dashed diagonal line looks like it might be suggesting a gusset, but it is not actually defined.

5. The dashed line is unclear

The dashed diagonal line could mean several things:

It might be a hidden edge, a rib, a proposed gusset, a centerline, or just a sketch mark.

Right now, it is ambiguous. If it is a gusset, it needs dimensions, thickness, and placement. If it is not part of the design, remove it.

Structural issues
6. The bracket is likely weak in bending

As drawn, the vertical leg is long and the base is thin. If the vertical wall is really 0.3 mm thick, the bracket would be extremely flexible and likely snap or delaminate.

For PLA, this is especially important because PLA is stiff but brittle. It does not like impact, high stress concentration, or long term creep under load.

Recommended improvements:

Increase wall thickness

Add a large internal fillet

Add triangular gussets on one or both sides

Print orientation should be specified

7. Print orientation is not specified

For FDM, print direction matters a lot. Layer lines can become the failure plane.

If the bracket is printed flat on its side, the L shape may be stronger in one loading direction but weaker in another. If printed upright, the vertical leg may separate along layer lines.

The drawing should include something like:

Recommended print orientation: bracket lying on side, holes vertical through Z, 100% infill or 5+ walls

or whatever orientation is intended.

8. No loading direction or design intent

For a bracket, the critical question is: what force is it carrying?

The sketch does not show:

load direction
mounting surface
supported component
expected force or weight
factor of safety
whether the holes are for bolts, screws, or pins

Without that, it is hard to judge whether PLA or the geometry is adequate.

Hole and annotation issues
9. Hole locations are underdefined

There are two holes, both apparently Ø4 mm, but the hole positions are not fully defined.

The 32 mm dimension appears to measure the spacing between holes, but it is unclear whether it is:

center to center
edge to edge
between tangent points
from one hole center to another

It should explicitly be center to center.

Also missing:

distance from left edge to first hole center
distance from bottom edge to hole centerline
distance from right edge to second hole center
hole pattern symmetry or centerline location

A better annotation would be:

2X Ø4.0 THRU
Hole spacing: 32 mm C-C
Hole centerline: 8 mm from bottom edge
First hole center: 25 mm from left edge

or similar.

10. Hole diameter may be too small for actual fasteners

A Ø4 mm hole is nominally for an M4 fastener, but if the part is FDM printed, a printed Ø4 mm hole may come out undersized.

For an M4 clearance hole, use something closer to:

Ø4.3 mm to Ø4.5 mm

If the screw is self-tapping into plastic, then it should be called out differently.

11. Hole symbols are drawn incorrectly for a technical drawing

The little plus signs inside the circles look like center marks, but they are oversized and visually cluttered. Center marks are fine, but the holes should be cleanly dimensioned with proper leaders or centerlines.

The callout should be something like:

2X Ø4.5 THRU

not just Ø4 mm floating near one hole.

Tolerance and drawing standard issues
12. ±0.001 in tolerance is wildly too tight for FDM

±0.001 in is about ±0.0254 mm.

That is much tighter than typical FDM printing accuracy, especially for PLA, holes, thin walls, and unsupported features.

For FDM, more realistic general tolerances would be around:

±0.2 mm to ±0.5 mm

For critical hole diameters, you may need to drill or ream after printing.

13. Overall height dimension is in inches but should probably be metric

The height is shown as 2.36 in, which equals about 59.9 mm.

Since the base length is 80 mm, the height should probably be labeled:

60 mm

Mixing 2.36 in with 80 mm makes the drawing look unfinished and easy to misread.

14. No origin or datum scheme

For a simple sketch, this might be acceptable, but for a real drawing, you want datums or at least consistent dimensioning from functional edges.

For example:

Datum A: bottom face
Datum B: left face
Datum C: rear face or center plane

Then hole locations can be dimensioned from those.

Geometry issues
15. Base height/thickness is not dimensioned

The horizontal leg has an apparent rectangular thickness, but no dimension is shown for the vertical height of the base section.

Same with the vertical leg width.

You need dimensions for:

vertical leg width/thickness
horizontal leg thickness
overall height
overall length
inside corner location
outer corner radius/fillet if any
part depth into page

Right now, only the overall base length and rough height are shown.

16. The 32 mm dimension is floating

The 32 mm dimension is above the base, but the extension lines do not clearly attach to hole centers or edges.

This is bad practice because someone could misinterpret it.

Use proper extension lines from the two hole centers.

Material/print callout issues
17. “Material: PLA – FDM print” is incomplete

That is a decent start, but for an actual printed part you may want:

material grade or filament type
layer height
infill percentage
wall/perimeter count
print orientation
nozzle size if wall thickness matters
post-processing requirements
support requirements
hole drilling requirement

Example:

Material: PLA, FDM printed
Layer height: 0.2 mm
Walls: 4 perimeters minimum
Infill: 40% gyroid minimum
Print orientation: flat on side
Drill holes to final size after printing

Cleaned-up version of what the drawing should include

At minimum, I would revise the sketch with these callouts:

Overall length: 80 mm
Overall height: 60 mm
Base thickness: maybe 8 mm
Vertical leg thickness: maybe 8 mm
Part width/depth: maybe 15 to 25 mm
Internal fillet: R5 mm minimum
Holes: 2X Ø4.5 THRU for M4 clearance
Hole spacing: 32 mm center to center
Hole centerline height: dimension from bottom edge
First hole center location: dimension from left edge
General tolerance: ±0.25 mm or ±0.5 mm
Material: PLA, FDM
Print settings/orientation: specified
Remove or define dashed diagonal line

Biggest “this will fail” issue: the 0.3 mm wall. For FDM PLA, that is the first thing I would fix. That bracket needs real wall thickness and probably a gusset if it is carrying load.

### Exchange 2 — OpenSCAD render (openscad_render.png)

**Host used:** _..._chatgpt

**Prompt I sent:**
You are reviewing a hand sketch for an L-bracket that will be 3D printed in PLA. Identify any issues you see which is dimensional, manufacturing, structural, or annotation. Be specific.
> _Same prompt as Exchange 1, applied to the render image._

**Screenshot:** `section4_screenshots/render_review.png`
![alt text](image-2.png)

**Response (copied from host):**

This v2 bracket is much better structurally than the original sketch because it now has actual printable thickness and a triangular gusset, but there are still several issues that would make it hard to manufacture, inspect, or trust as a structural PLA bracket.

1. Dimensional issues
No actual dimensions are shown

The render says OpenSCAD render bracket_v2, but the part itself has no dimension callouts visible.

Missing critical dimensions:

Feature	Missing info
Overall base length	Need total horizontal length
Overall height	Need vertical leg height
Base thickness	Need thickness of horizontal plate
Upright thickness	Need wall thickness of vertical leg
Part width/depth	Need extrusion depth into the page
Gusset size	Need length, height, and thickness
Back/top tab	Need dimensions and purpose
Hole locations	No mounting holes are visible or dimensioned

Right now this is more of a concept render than a usable engineering drawing.

2. Hole issue: original holes disappeared

The hand sketch had two Ø4 mm holes in the base. In this v2 render, I do not see those holes at all.

That is a big problem if this is meant to mount to something.

You should add and call out:

2X Ø4.5 mm THRU

or, if using heat-set inserts or self-tapping screws, call that out specifically.

For M4 fasteners in FDM PLA, I would not use exactly Ø4.0 mm unless you plan to drill them after printing. A printed Ø4.0 mm hole will often come out undersized.

Better callout:

2X Ø4.5 THRU FOR M4 CLEARANCE
HOLES TO BE DRILLED TO FINAL SIZE AFTER PRINTING

3. Manufacturing issues
PLA is okay for a prototype, but not great for sustained load

PLA is stiff and easy to print, but it is brittle and can creep under load, especially if the bracket stays loaded over time or gets warm.

For a class prototype, PLA is fine. For a working bracket, I would consider:

PETG for toughness
PLA+ for better impact resistance
Nylon or carbon fiber nylon for a stronger functional bracket

Print orientation is not specified

This matters a lot. If the bracket is printed in the wrong orientation, the upright can snap along layer lines.

The drawing should state something like:

Print orientation: base flat on build plate

That is probably the easiest orientation, but the vertical wall and gusset will grow upward in Z, meaning loads that try to peel the upright away from the base may split between layers.

A stronger print might require printing it on its side, depending on load direction, but that could need supports.

No wall/perimeter/infill specification

The render says:

0.4 mm nozzle, 0.2 mm layers

Good start, but not enough.

Add:

Walls: 4 minimum
Top/bottom layers: 5 minimum
Infill: 40 to 60 percent minimum
Infill pattern: gyroid or cubic
Material: PLA or PLA+

For a structural bracket, wall count often matters more than infill.

4. Structural issues
The triangular gusset is good, but it needs better definition

The gusset is a smart improvement. It will reduce bending at the inside corner and help prevent the upright from flexing.

But it needs to be defined:

Gusset thickness
Gusset height
Gusset length along base
Whether there is one gusset or two side gussets
Whether it is centered or flush to one side

From the render, it looks like one central triangular rib. That helps, but for a bracket with width, two side gussets may be stronger and easier to inspect.

Sharp inside corners still appear present

The inside base to upright transition looks sharp. Sharp corners create stress concentrations.

Add fillets:

Inside corner fillet: R3 to R6 mm
Gusset root fillets: R2 to R4 mm

This will improve strength and reduce crack initiation.

Top/back tab is unclear

There is a small horizontal rectangular tab sticking out near the top rear of the vertical leg. It is not explained.

Questions that need to be answered:

Is it a mounting tab?
Is it support material?
Is it part of the bracket?
Does it interface with another component?
Does it need holes?

Right now it looks accidental or underdefined. If it is functional, dimension it and label it. If not, remove it.

Base may be too long and unsupported depending on load

The base projects far outward. If the load is applied near the far end, the base itself may bend. The gusset mostly strengthens the vertical to base corner, but it does not reinforce the far end of the base.

If the far end carries load, consider:

thicker base
ribs along the base
mounting holes farther apart
larger fillets
two gussets instead of one

5. Annotation issues
Render title is not enough

The title says:

OpenSCAD render bracket_v2

That is fine as an internal label, but a proper drawing needs:

Part name
Revision
Material
Units
Scale
Tolerance
Designer/date
Print settings
Drawing view labels

Example:

L-Bracket, Rev B
Units: mm
Material: PLA, FDM printed
General tolerance: ±0.3 mm unless otherwise noted

No tolerance is given

The previous sketch had ±0.001 in, which was way too tight for FDM. This render has no tolerance shown.

For FDM PLA, a more realistic general tolerance would be:

±0.25 mm for normal dimensions
±0.5 mm for noncritical exterior dimensions
Hole diameters to be drilled/reamed after print if critical

Units are not stated

The render mentions nozzle and layer height in mm, but the actual part dimensions are not shown. The drawing should explicitly say:

All dimensions in millimeters unless otherwise noted

6. Specific fixes I would make

For v3, I would update it like this:

Area	Recommended fix
Units	State all dimensions in mm
Tolerance	Use ±0.25 mm or ±0.5 mm, not ±0.001 in
Holes	Add the two base holes back
Hole size	Use Ø4.5 mm through for M4 clearance
Hole locations	Dimension from datum edges and center to center
Gusset	Dimension height, length, and thickness
Corners	Add fillets at inside corner and gusset roots
Print settings	Add walls, infill, layer height, orientation
Material	PLA okay for prototype, PLA+ or PETG better for function
Top tab	Define its purpose or remove it
Drawing	Add front, side, top, and isometric views
Biggest remaining problems

The three biggest issues I see are:

No dimensions are visible, so the part is not manufacturable from the drawing.
The mounting holes from the original sketch are missing.
The gusset is good, but it is not dimensioned or filleted.

The geometry is definitely moving in the right direction, though. The gusset is a big upgrade over the first version. Now it needs to become a proper drawing instead of just a render.

### Exchange 3 — Text-only baseline (no image)

**Host used:** _..._chatgpt

**Prompt I sent:**
I have a hand-sketched L-bracket for 3D printing in PLA. Annotated dimensions, two M4 holes, gusset on the back. What issues should I check for?

**Screenshot:** `section4_screenshots/textonly_review.png`
![alt text](image-3.png)
**Response (copied from host):**

1. Units and tolerances

Make sure the drawing uses one main unit system, preferably mm.

Avoid something like:

80 mm length
2.36 inch height
±0.001 inch tolerance

That gets messy fast. For FDM PLA, ±0.001 inch is way too tight. A better general note would be:

All dimensions in mm unless noted. General tolerance ±0.3 mm.

For noncritical dimensions, even ±0.5 mm is usually more realistic.

2. Wall thickness

Do not use a 0.3 mm wall for a 0.4 mm nozzle. That is thinner than a normal extrusion line and may not print correctly.

For PLA FDM, use at least:

1.2 mm minimum wall thickness, about 3 nozzle widths
2 to 4 mm for a functional bracket
4 or more perimeters if it carries load

3. M4 holes

If the holes are for M4 bolts, do not make them exactly Ø4.0 mm unless you plan to drill them after printing. Printed holes usually come out undersized.

Use:

2X Ø4.5 mm THRU for M4 clearance

or:

2X Ø4.0 mm pilot, drill to Ø4.5 mm after printing

Also check that the hole locations are fully defined:

center to center spacing
distance from one edge to first hole center
distance from bottom edge to hole centerline
edge distance around each hole

4. Hole edge distance

Do not place M4 holes too close to the edge. PLA can crack around the holes.

A decent minimum is:

2D from hole center to edge

For a Ø4.5 mm hole, that means about:

9 mm minimum from hole center to edge

More is better if the bracket carries load.

5. Missing third dimension

A side sketch is not enough. You need the bracket width or depth into the page.

Include:

overall width/depth
base thickness
upright thickness
gusset thickness

Without that, the part cannot be modeled or inspected correctly.

6. Gusset definition

The gusset is good, but it needs to be fully specified.

Check:

gusset height
gusset length along base
gusset thickness
whether it is centered or on both sides
whether it interferes with bolts, washers, or the mounted part

For a stronger bracket, two side gussets are often better than one thin center gusset.

7. Fillets and stress concentrations

Sharp inside corners are bad for brackets. Add fillets where the vertical leg meets the base and where the gusset meets the bracket.

Good starting values:

inside L corner: R3 to R6 mm
gusset root fillets: R2 to R4 mm
outside edges: R1 to R2 mm

This improves strength and makes the part less likely to crack.

8. Print orientation

Call out the intended print orientation. This matters because layer lines are usually the weak direction.

A simple note could be:

Print orientation: base flat on build plate

That is easy to print, but the upright may be weaker against loads that peel it away from the base. If the bracket will see serious bending load, consider printing on its side or redesigning so layer lines do not split at the corner.

9. Print settings

For a functional PLA bracket, include basic print settings:

0.4 mm nozzle
0.2 mm layer height
4 to 6 walls
5 or more top/bottom layers
40 to 60 percent infill
gyroid, grid, or cubic infill
PLA+ or PETG preferred if loaded

Wall count matters more than infill for many bracket designs.

10. PLA material limitations

PLA is fine for a prototype, but check the environment. PLA can creep under constant load and soften in hot areas.

Avoid PLA if the bracket is:

near heat
inside a car
under constant tension or bending
holding something safety critical
exposed to impact

For a working bracket, PETG, PLA+, nylon, or carbon fiber nylon may be better.

11. Annotation clarity

Check that every callout is unambiguous.

Use notes like:

2X Ø4.5 THRU
32 mm C to C
R4 TYP
4 mm THK
All dimensions in mm

Avoid floating dimensions with no clear extension lines. The hole spacing should clearly reference the hole centers, not the edges of the circles.

12. Fastener and washer clearance

Make sure there is enough flat area around the holes for the screw head, washer, or nut.

For M4, check:

washer outside diameter
socket head cap screw diameter
nut clearance if using a nut
tool access for tightening

A hole may fit the bolt but still fail because the washer overlaps an edge or hits the upright.

Quick checklist

For your next version, I would make sure the drawing has:

Units: mm
General tolerance: ±0.3 mm
Overall length and height
Part width/depth
Base and upright thickness
2X Ø4.5 THRU for M4 clearance
Hole spacing and edge distances
Gusset dimensions
Fillets at inside corner and gusset roots
Print orientation and print settings
Material note: PLA prototype, PLA+/PETG recommended for functional load

The biggest things to fix are usually: hole sizing, missing depth dimension, unrealistic tolerance, weak layer orientation, and sharp inside corners.

### Comparison table

Fill in the issue catalog (your own ground-truth list of what's wrong
with the bracket — you should be able to identify 3–5 distinct issues
just from the sketch and render). Then count what each AI exchange
caught.

**My ground-truth issue list:**

1. **±0.001" tolerance is well below FDM capability.** The sketch
   states "ALL DIMS ±0.001" unless noted" — that's ±0.025 mm, and
   a 0.4 mm nozzle / 0.2 mm layer FDM print holds more like
   ±0.1-0.2 mm. The blanket tolerance is unachievable.

2. **0.3 mm wall thickness is too thin.** The red callout on the
   sketch labels a 0.3 mm wall — less than one nozzle width on a
   0.4 mm nozzle, so the slicer will either drop it or print a
   fragile single-line wall.

3. **Mixed-unit annotations.** The sketch gives the bracket height
   as 2.36" but every other dimension (80 mm length, 32 mm hole
   spacing, 4 mm hole, 0.3 mm wall) is in millimeters. Mixing US
   customary and SI on the same drawing is a red flag.

4. **No gusset / no fillet at the inside corner.** Sharp 90° inside
   corner with no fillet or rib at the highest-stress point. The
   dashed line in v1 suggests a gusset was considered but it isn't
   dimensioned and the v2 render shows no gusset implemented.

5. **Hole undersized for an M4 fastener.** The hole callout reads
   "4 mm" but an M4 clearance hole should be ~4.3-4.5 mm — after
   print-shrinkage a nominal 4 mm hole would bind.

| Exchange                | Host | Issues caught | Out of |
|-------------------------|------|---------------|--------|
| Sketch image            |chatgpt|   17         |    5   |
| OpenSCAD render image   |chatgpt|        6     |    6   |
| Text-only baseline      |chatgpt|        12    |    5   |

### ✏️ YOUR OBSERVATION

*In 3 sentences:* Where does vision help? Where does it fall short?
What would you NOT trust to a vision review?

Vision helped most on the issues that were drawn explicitly — the 0.3 mm wall callout, the sharp inside corner, the mixed-unit
dimensions — because those are visible at a glance. It fell short on numbers it had to actually read, especially the
±0.001" tolerance note at the bottom of the sketch; in my exchanges the [sketch / render / text-only] version was the only
one to flag it specifically as an FDM problem. I wouldn't trust a vision review for anything that depends on reading exact text
inside the drawing — tolerances, thread callouts, material specs those should be transcribed and re-checked in a text-mode
follow-up.

> **Forward connection.** In Part B you'll do this exact workflow on
> your own MiniClaw CAD work — three documented checkpoints, a
> screenshot to your AI host, a documented review. Section 4 was the
> warm-up; Part B is the real thing.

---
## Section 5: Composition Capstone — A Skill That Uses Multiple Tools [8 pts]

**Concept.** A real engineering AI workflow combines multiple
capabilities. The capstone exercise: write a skill that orchestrates
the AI to consult relevant project history (your **RAG tool from
Section 3**) *and* handle unit conversions (your **converter from
Section 1**) in the same response, then produce a synthesized
engineering answer drawing on both. The AI decides which tool to call
when, guided by the skill.

### What to build

1. Write `skills/engineering_question_with_context.md`. The skill
   should instruct the AI to:
   - First query the RAG for relevant project history when the
     question touches Ridgeline projects
   - Use the unit converter whenever a quantity in one unit system
     needs to be expressed in another
   - **Synthesize:** tie retrieved context to the specific question,
     don't dump retrieved chunks
2. Load the skill as system prompt. Make BOTH the RAG tool and the
   unit converter available.
3. Run a composite query through `run_with_logging()`:

   > *"I'm reviewing a bracket design for a project similar to your
   > CardioSense work. The bracket needs to handle a 500 lbf load.
   > What did Ridgeline use for the CardioSense enclosure, and what's
   > that load in newtons?"*
4. **Without the skill:** rerun the same query with only the tools
   (no system prompt). Compare. What did the skill change?

### Key concepts

- This is the agent loop with multiple tools — the architectural
  pattern that powers Claude Code, Copilot agent mode, Cursor agent,
  and every "AI coworker" product on the market.
- The skill is what turns *"AI with tools"* into *"AI with judgment
  about which tools to use when."* Without the skill, the AI may
  answer from training without consulting the RAG, may convert units
  mentally, or may not synthesize across both.
- Composition introduces the failure mode of *under-consultation* —
  the AI deciding it doesn't need a tool when it does. The skill
  counters this by making consultation mandatory in the workflow.

### Proof of work [8 pts]

The text of `engineering_question_with_context.md`, two transcripts
(with skill / without skill), a summary table, and a 3-sentence
observation: *"What does a skill add to a tool-equipped AI? What's
the limit of what skills can do?"*

### ✏️ YOUR TURN — Write `skills/engineering_question_with_context.md`

Same format as your Section 2 skill (front matter, when-to-use, steps,
what-to-flag, what-not-to-do). Include explicit instructions about
*when* to call each tool.

In [32]:
# Step 1 — Print your composition skill so it's part of the cell output.
composite_skill_path = Path("skills/engineering_question_with_context.md")
if not composite_skill_path.exists():
    composite_skill_path = Path("MP3/Part A/skills/engineering_question_with_context.md")

print("=" * 62)
print(f"{composite_skill_path.name}")
print("=" * 62)
if composite_skill_path.exists():
    print(composite_skill_path.read_text(encoding="utf-8"))
else:
    print("(file does not exist yet — create it, then re-run this cell)")

engineering_question_with_context.md
name: engineering-question-with-context
description: >
Answer an engineering question that mixes a Ridgeline-internal
reference with a quantitative calculation. Orchestrates two tools:
query_ridgeline_rag for project history, billing, and internal
standards, and convert_units for unit-system translation. The goal
is one synthesized engineering answer, not retrieved chunks pasted
next to a raw converted number.
Skill: Engineering Question with Project Context
When to use this skill
Use this when the user's question combines two or more of:

a reference to a specific Ridgeline project, billing rate, or
internal standard
a numeric quantity that may need converting between US customary
and SI
a comparison between something Ridgeline does and broader
engineering practice.

If the question is purely general engineering knowledge with no
Ridgeline reference, don't call the RAG. If there's no number in the
question, don't call the converter. Don't call tool

In [33]:
# Step 2 — WITH skill: load skill as system prompt, expose both tools.
#
composite_query = (
    "I'm reviewing a bracket design for a project similar to your "
    "CardioSense work. The bracket needs to handle a 500 lbf load. "
    "What did Ridgeline use for the CardioSense enclosure, and what's "
    "that load in newtons?"
)

both_tools = [rag_tool_good, convert_units_tool]
both_handlers = {
    "query_ridgeline_rag": query_ridgeline_rag,
    "convert_units":       convert_units,
}

result_with_skill = run_with_logging(
    messages=[{"role": "user", "content": composite_query}],
    tools=both_tools,
    tool_handlers=both_handlers,
    system=composite_skill_path.read_text(encoding="utf-8"),
    label="section5_with_skill",
)

══════════════════════════════════════════════════════════════
[USER]
I'm reviewing a bracket design for a project similar to your CardioSense work. The bracket needs to handle a 500 lbf load. What did Ridgeline use for the CardioSense enclosure, and what's that load in newtons?
══════════════════════════════════════════════════════════════
[ITERATION 1]
[TOOL CALL] query_ridgeline_rag
  Args: {"question": "CardioSense enclosure bracket material selection and load capacity"}
[TOOL RESULT]
  {
    "chunks": [
      {
        "text": "MATERIAL SELECTION REPORT\nProject: Duraline Automotive Rear Suspension Bracket Redesign\nProject Number: REP-2024-055\nAuthor: Dr. Sarah Hendricks, PE, PhD\nDate: October 2024\n\nOBJECTIVE\nSelect a replacement material for the Duraline RA-800 rear suspension bracket, currently manufactured from SAE 1045 hot-rolled steel. The client's objectives are a 25% weight reduction while maintaining equivalent fatigue life (minimum 500,000 cycles at peak load) and s

In [34]:
# Step 3 — WITHOUT skill: same query, same tools, no system prompt.
result_without_skill = run_with_logging(
    messages=[{"role": "user", "content": composite_query}],
    tools=both_tools,
    tool_handlers=both_handlers,
    label="section5_without_skill",
)

══════════════════════════════════════════════════════════════
[USER]
I'm reviewing a bracket design for a project similar to your CardioSense work. The bracket needs to handle a 500 lbf load. What did Ridgeline use for the CardioSense enclosure, and what's that load in newtons?
══════════════════════════════════════════════════════════════
[ITERATION 1]
[TOOL CALL] query_ridgeline_rag
  Args: {"question": "What material was used for the CardioSense enclosure bracket in the project documentation?"}
[TOOL RESULT]
  {
    "chunks": [
      {
        "text": "eed 28 grams.\n\nMATERIAL SELECTION AND DESIGN\nAfter evaluating six candidate materials, Ridgeline selected glass-filled nylon PA66-GF30 for the enclosure body. This material provides the required mechanical strength (tensile strength 185 MPa), dimensional stability, and chemical resistance while meeting the weight target. The glass-filled nylon also offers excellent moldability for the thin-wall sections (minimum wall thickness 0.8

In [36]:
# Step 4 — Summary table.
#
def count_tool(result, name):
    n = 0
    for tc in result["tool_calls_made"]:
        if tc["name"] == name:
            n += 1
    return n

print("Run             rag_called   converter_called")
print("-" * 50)
print(f"With skill      {count_tool(result_with_skill, 'query_ridgeline_rag')}            {count_tool(result_with_skill, 'convert_units')}")
print(f"Without skill   {count_tool(result_without_skill, 'query_ridgeline_rag')}            {count_tool(result_without_skill, 'convert_units')}")

Run             rag_called   converter_called
--------------------------------------------------
With skill      1            1
Without skill   1            1


### ✏️ YOUR OBSERVATION

*In 3 sentences:* What does a skill add to a tool-equipped AI? What's
the limit of what skills can do?

Both runs called both tools (1 RAG, 1 converter each), so the skill didn't change which tools got called on this query. What it changed was the form of the answer and with the skill the response wove the ~2.22 kN value into one sentence and cited the doc_id; without it the answer read like two separate replies stapled together. What a skill can't do is fix bad retrieval and if the RAG returns the wrong chunks, the skill can tell the AI to flag the gap but can't conjure a chunk that isn't in the corpus.

> **Forward connection.** Part B asks you to do this on your own
> engineering work — write skills, expose MiniClaw RAG as an MCP
> tool, and run a documented design review. The composition you just
> built is the seed of that workflow at MiniClaw scale.

---
## Section 6: Reflections [8 pts]

Three short reflections. 3–4 sentences each, with at least one
concrete engineering example where it makes sense. The grader is
looking for thought, not length.

### Reflection 1 — Where do tools end and skills begin?

In your own words, when would you reach for a new *tool* vs. a new
*skill*? Give a small engineering example for each.

I'd write a new tool when the work is deterministic and worth auditing — a unit conversion, a RAG retrieval, a CAD parameter query — because a tool produces a citable value that shows up as a [TOOL CALL] line in the transcript. I'd write a new skill when the work is judgmental and needs to be reused, like deciding when a task should pull from company context versus answer from training. Concrete split from this notebook: convert_units is a tool because converting 25.4 mm to inches is arithmetic, and unit_check_skill is a skill because deciding what counts as a "category error" is judgment that lives in writing, not in code.

### Reflection 2 — What's the durable lesson?

MCP, function calling, GitHub Models, host-based vision review —
these are 2026 specifics. What's the underlying idea you'd carry
forward if all of these were replaced next year?

The protocol-specific stuff like OpenAI tool schema syntax, GitHub Models endpoints, the MCP wire format will all be replaced inside a year or two and that's fine. The idea that sticks is the separation between capability (tools the AI can call), judgment (skills the AI consults), and orchestration (the agent loop that decides which to invoke when). Section 5 made this concrete for me: same RAG plus converter, same query, same model, but the with-skill and without-skill runs produced meaningfully different answers and the orchestration layer was doing real work even when the tool-call counts came out the same.

### Reflection 3 — What worries you?

When AI orchestrates tools and consults skills, the engineer steps
back from doing the work directly. What's the failure mode you'd
watch for in a real engineering team?

The failure mode I'd watch for is under-consultation dressed up as confidence and the AI deciding it doesn't need the RAG and fabricating a Ridgeline practice that sounds right because it matches industry-standard language. The vague-description run in Section 3 was a small version of this: same model, same question, same tool available, and the AI still returned the right rating factor only because the user's question explicitly named "Millbrook County bridge." On a real team that pattern would mean a junior engineer asking about company practice and getting back something authoritative-sounding that drifts from what the company actually does, with no [TOOL CALL] line to flag that retrieval was skipped.

---

## Submission checklist

Before pushing your final commit, verify:

- [ ] All cells in Sections 1, 2, 3, 5 run end-to-end without errors
- [ ] At least one `[TOOL CALL]` line is visible in each of Sections 1, 3, 5
- [ ] `skills/unit_check_skill.md` and
      `skills/engineering_question_with_context.md` both exist and were
      printed as part of the cell output
- [ ] Section 4 has three documented exchanges with screenshots
      committed in `section4_screenshots/`
- [ ] `transcripts/` contains the JSON files written by
      `run_with_logging()` (these get committed automatically when you
      push)
- [ ] Section 6 reflections are filled in

> **Codespaces save warning.** Make sure the notebook is saved
> (`Ctrl+S`) and committed via Source Control before you stop your
> Codespace. Unstaged changes are lost when the container is deleted.